# Discriminant Analysis (LDA)

## Overview
Linear Discriminant Analysis (LDA) finds linear combinations of predictor variables that best separate known groups. It serves two related purposes:

| Purpose | Goal | Key output |
|---|---|---|
| **Description** | Which variables discriminate groups? | Discriminant function coefficients, structure matrix |
| **Prediction** | Classify new observations into groups | Posterior probabilities, cross-validated accuracy |

**Relationship to MANOVA:** MANOVA asks *whether* groups differ multivariate. LDA asks *how* they differ and classifies new cases. The canonical discriminant scores plotted after MANOVA are derived directly from LDA. When used descriptively, LDA and MANOVA are two views of the same analysis.

**Assumptions:** multivariate normality within groups; homogeneous within-group covariance matrices (same as MANOVA); linear decision boundaries. For heterogeneous covariances, use Quadratic Discriminant Analysis (QDA).

**Reference:** Quinn & Keough (2002) ch. 16

---

In [ ]:
library(tidyverse); library(MASS); library(caret)
set.seed(7)
# Fish morphometrics across 3 lake populations
# Variables: body_depth, head_length, eye_diameter, pectoral_fin
pop <- rep(c("lake_A", "lake_B", "lake_C"), each = 40)
mu_list <- list(
  lake_A = c(42, 18, 6.2, 14),
  lake_B = c(38, 20, 5.5, 16),
  lake_C = c(45, 17, 7.0, 13)
)
Sigma <- matrix(c(9,2,1,1.5, 2,4,0.8,0.9,
                   1,0.8,1,0.4, 1.5,0.9,0.4,2), 4, 4)
dat <- do.call(rbind, lapply(pop, function(p)
  MASS::mvrnorm(1, mu_list[[p]], Sigma))) |>
  as.data.frame() |>
  setNames(c("body_depth", "head_length", "eye_diameter", "pectoral_fin"))
dat$population <- factor(pop)
cat("n =", nrow(dat), "fish across", nlevels(dat$population), "populations\n")
print(dat |> group_by(population) |> summarise(across(body_depth:pectoral_fin, \(x) round(mean(x),1))))

---
## Descriptive LDA: discriminant functions

In [ ]:
# Fit LDA (prior = proportional to group sizes)
lda_fit <- lda(population ~ body_depth + head_length + eye_diameter + pectoral_fin,
               data = dat)

cat("=== LDA Summary ===\n")
print(lda_fit)

# Proportion of between-group variance on each discriminant axis
cat("\nProportion of trace (variance explained by each LD):\n")
prop <- lda_fit$svd^2 / sum(lda_fit$svd^2)
for (i in seq_along(prop))
  cat(sprintf("  LD%d: %.1f%%\n", i, prop[i] * 100))

In [ ]:
# Structure matrix: correlations between original variables and each LD
# More interpretable than raw coefficients (which are standardised)
cat("--- Structure matrix (variable-LD correlations) ---\n")
pred_scores <- predict(lda_fit)$x
struct_mat <- cor(dat[, 1:4], pred_scores)
print(round(struct_mat, 3))
cat("(Correlations > |0.30| are typically considered meaningful)\n")

# Biplot: discriminant scores coloured by population
scores_df <- as.data.frame(pred_scores)
scores_df$population <- dat$population
ggplot(scores_df, aes(LD1, LD2, colour = population)) +
  geom_point(alpha = 0.75, size = 2.5) +
  stat_ellipse(level = 0.68, linewidth = 0.9) +
  geom_text(data = as.data.frame(lda_fit$means %*% lda_fit$scaling) |>
              rownames_to_column("population"),
            aes(LD1, LD2, label = population), colour = "black",
            fontface = "bold", size = 3.5, vjust = -1) +
  labs(title = "LDA: Discriminant scores by population",
       subtitle = sprintf("LD1 = %.0f%%, LD2 = %.0f%% of between-group variance",
                          prop[1]*100, prop[2]*100)) +
  theme_minimal() + theme(legend.position = "none")

---
## Predictive LDA: classification accuracy

In [ ]:
# Resubstitution (apparent) accuracy — OPTIMISTICALLY biased
pred_resub <- predict(lda_fit)$class
cat("Resubstitution confusion matrix:\n")
print(table(Predicted = pred_resub, Actual = dat$population))
cat("Resubstitution accuracy:",
    round(mean(pred_resub == dat$population) * 100, 1), "%\n")

# Leave-one-out cross-validated accuracy — unbiased estimate
lda_cv <- lda(population ~ body_depth + head_length + eye_diameter + pectoral_fin,
              data = dat, CV = TRUE)
cat("\nLeave-one-out CV confusion matrix:\n")
print(table(Predicted = lda_cv$class, Actual = dat$population))
cat("LOO-CV accuracy:",
    round(mean(lda_cv$class == dat$population) * 100, 1), "%\n")

# Posterior probabilities for first 5 observations
cat("\nPosterior probabilities (first 5 obs):\n")
print(round(head(predict(lda_fit)$posterior, 5), 3))

In [ ]:
# QDA when covariance matrices are heterogeneous
qda_fit <- qda(population ~ body_depth + head_length + eye_diameter + pectoral_fin,
               data = dat, CV = TRUE)
cat("QDA LOO-CV accuracy:",
    round(mean(qda_fit$class == dat$population) * 100, 1), "%\n")
cat("(QDA fits separate covariance per group — more flexible, needs more data)\n")

# When to use LDA vs QDA:
cat("\nLDA vs QDA decision guide:\n")
cat("  LDA: equal covariances, fewer parameters, better with small n\n")
cat("  QDA: heterogeneous covariances; use Box's M test (from MANOVA) to check\n")
cat("  Regularised DA (rda in klaR): compromise for high-dim data\n")

---
## Common Pitfalls

**1. Confusing description and prediction goals**
Descriptive LDA (structure matrix, biplot) identifies which variables discriminate groups in your sample. Predictive LDA (confusion matrix, accuracy) estimates classification error for new observations. Resubstitution accuracy is always optimistic — always report LOO-CV or test-set accuracy for honest performance estimates.

**2. Interpreting raw discriminant coefficients instead of the structure matrix**
Raw (or standardised) coefficients are analogous to partial regression coefficients — they are affected by correlations among predictors. The structure matrix (correlations between each original variable and each LD) is more interpretable and more stable.

**3. Using LDA when group sizes are very unequal**
With very different group sizes, the prior probability matters greatly. The default (`prior = proportional`) reflects the sample composition but may not represent the true population. Set priors explicitly if you have better information (e.g., equal priors for classification).

**4. Not checking assumption of equal covariance matrices**
LDA assumes within-group covariance matrices are homogeneous. Use Box's M test or compare LDA vs QDA accuracy. If QDA is substantially better, covariance heterogeneity is real and LDA results are unreliable.

**5. Applying LDA when p ≥ n within groups**
LDA requires inverting the within-group covariance matrix. With more variables than observations per group, this fails. Use regularised discriminant analysis (`klaR::rda`) or PCA-then-LDA as a preprocessing step.

**6. Treating classification accuracy as an absolute measure**
Accuracy depends on the number of classes and prior probabilities. Always compare against a null classifier (random or always-predict-majority). A 60% accuracy with 3 equal-sized classes is better than it sounds (null = 33%); with 2 classes where one is 90% of data, 60% would be worse than the null.


---
*r_methods_library - Samantha McGarrigle*